# Spark Compass — **full-stack console** (08)

Run this notebook **on the cluster head** (Spark NICs, Docker, `spark-vllm-docker`). Not for your laptop.

**Prefer stepping through milestones?** Use the numbered notebooks **`01`–`07`** in this folder — see [`README.md`](README.md).

This file runs **all gates in one place** (same content as the original single-notebook wizard).

```bash
cd /path/to/dgx-spark-cluster-compass/wizard
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements-wizard.txt
jupyter lab 08_full_stack_console.ipynb
```

Upstream workhorses (install separately on the Sparks):
- [NVIDIA connect-two-sparks playbook](https://github.com/NVIDIA/dgx-spark-playbooks/tree/main/nvidia/connect-two-sparks)
- [eugr/spark-vllm-docker](https://github.com/eugr/spark-vllm-docker)


---
## Configuration — edit these values

Match your playbook / `launch-cluster.sh` flags.

In [ ]:
# Cluster network (example layout)
HEAD_IP = "10.0.0.1"
WORKER_IP = "10.0.0.2"

# Interfaces (typical Spark naming — yours may differ)
ETHERNET_IF = "enp1s0f0np0"
IB_IF = "rocep1s0f0"  # RoCE / IB device name prefix for show_gids

# SSH to worker for remote checks (passwordless key per NVIDIA playbook)
SSH_USER = "cesarb-ai"  # change to your cluster user
WORKER_SSH = f"{SSH_USER}@{WORKER_IP}"

# eugr repo root on this machine
SPARK_VLLM_DOCKER = "~/spark-vllm-docker"

# Model path *inside* the container when using hub materialized layout
MODEL_ID = "/root/.cache/huggingface/materialized/Qwen-Qwen2.5-7B-Instruct"

# Optional: docker container name for log tail (set after you know it)
VLLM_CONTAINER_NAME = "vllm_node"  # eugr default; adjust if different

# Gates — set False to pause before destructive / long-running steps
RUN_LAYER1_GATE = True
RUN_LAYER2_GATE = True
RUN_LAYER3_GATE = True
RUN_ZOMBIE_CHECK = True
RUN_LAUNCH_CLUSTER = False  # keep False until you are ready; then True
RUN_HEALTH_MONITOR = False  # True after cluster is up

---
## Layer 1 — Physical & network heartbeat

If the link is not **UP** or `ping` fails, no amount of vLLM tuning will help.

In [ ]:
import subprocess
import sys


def sh(cmd: list[str] | str, *, shell: bool = False) -> subprocess.CompletedProcess:
    if isinstance(cmd, str) and not shell:
        raise ValueError("Pass argv list or use shell=True")
    return subprocess.run(
        cmd,
        shell=shell,
        capture_output=True,
        text=True,
    )


def check_link(interface: str) -> bool:
    r = sh(["ip", "addr", "show", interface])
    if r.returncode != 0:
        print(f"❌ {interface}: not found or ip failed:\n{r.stderr}")
        return False
    if "state UP" in r.stdout:
        print(f"✅ {interface}: link UP")
        return True
    print(f"❌ {interface}: link not UP — check cabling / iface name.\n{r.stdout}")
    return False


def ping_host(host: str) -> bool:
    r = sh(["ping", "-c", "2", "-W", "2", host])
    ok = r.returncode == 0
    print(("✅" if ok else "❌") + f" ping {host}")
    if not ok:
        print(r.stdout + r.stderr)
    return ok


layer1_ok = check_link(ETHERNET_IF) and ping_host(WORKER_IP)
if not RUN_LAYER1_GATE:
    print("RUN_LAYER1_GATE is False — skipping hard stop")
elif not layer1_ok:
    print("\n⛔ Gate: fix Layer 1 before continuing.", file=sys.stderr)
else:
    print("\n✅ Layer 1 gate passed")

---
## Layer 2 — Blackwell / GPU symmetry

Tensor parallel expects **compatible** GPUs and drivers on **each** node. Mismatched driver versions or “mystery” VRAM on one side breaks planners and NCCL timing.

In [ ]:
def nvidia_csv() -> list[dict[str, str]]:
    r = sh(
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.total,driver_version",
            "--format=csv,noheader,nounits",
        ]
    )
    if r.returncode != 0:
        print(r.stderr)
        return []
    rows = []
    for line in r.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 4:
            rows.append(
                {
                    "index": parts[0],
                    "name": parts[1],
                    "mem_mib": parts[2],
                    "driver": parts[3],
                }
            )
    return rows


def remote_nvidia_csv() -> list[dict[str, str]]:
    r = sh(["ssh", WORKER_SSH, "nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv,noheader,nounits"])
    if r.returncode != 0:
        print(f"❌ ssh {WORKER_SSH} failed:\n{r.stderr}")
        return []
    rows = []
    for line in r.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 4:
            rows.append(
                {"index": parts[0], "name": parts[1], "mem_mib": parts[2], "driver": parts[3]}
            )
    return rows


print("=== Local GPUs ===")
local_rows = nvidia_csv()
for row in local_rows:
    print(row)

print("\n=== Remote GPUs (worker) ===")
remote_rows = remote_nvidia_csv()
for row in remote_rows:
    print(row)

layer2_ok = bool(local_rows and remote_rows)
if layer2_ok:
    ld, rd = local_rows[0]["driver"], remote_rows[0]["driver"]
    ln, rn = local_rows[0]["name"], remote_rows[0]["name"]
    lm, rm = local_rows[0]["mem_mib"], remote_rows[0]["mem_mib"]
    if ld != rd:
        print(f"\n⚠️ Driver mismatch: local {ld} vs worker {rd}")
        layer2_ok = False
    if ln != rn:
        print(f"\n⚠️ GPU name mismatch: local {ln} vs worker {rn}")
    if lm != rm:
        print(f"\n⚠️ VRAM total mismatch: local {lm} MiB vs worker {rm} MiB")

if not RUN_LAYER2_GATE:
    print("RUN_LAYER2_GATE is False — not enforcing gate")
elif not layer2_ok:
    print("\n⛔ Gate: resolve GPU/driver symmetry before NCCL/vLLM.", file=sys.stderr)
else:
    print("\n✅ Layer 2 gate passed (basic symmetry)")

---
## Layer 3 — RoCE / RDMA GID discovery

Wrong **GID index** is a classic silent killer for NCCL over RoCE. Many Spark setups use **RoCE v2** with a predictable row in `show_gids`; your site may differ — treat output as a hint, not law.

If `show_gids` is missing, install/use `rdma-core` tools from your distro or inspect GIDs under `/sys/class/infiniband/`.

In [ ]:
from pathlib import Path


def run_show_gids() -> str:
    r = sh(["show_gids"])
    if r.returncode == 0:
        return r.stdout
    r2 = sh("which show_gids", shell=True)
    return f"(show_gids failed: {r.stderr.strip() or 'not installed'})\nwhich: {r2.stdout.strip()}"


print("=== show_gids (filtered by RoCE device prefix) ===")
out = run_show_gids()
for line in out.splitlines():
    if IB_IF in line or "v2" in line.lower() or "roce" in line.lower():
        print(line)

print("\n=== sysfs GID sample (first port) ===")
gid_dirs = sorted(Path("/sys/class/infiniband").glob("*/ports/*/gids/*"))
for p in gid_dirs[:12]:
    try:
        val = p.read_text().strip()
    except OSError:
        continue
    if val and val != "0000:0000:0000:0000:0000:0000:0000:0000":
        print(f"{p}: {val}")

layer3_ok = "failed" not in out.lower() and "not installed" not in out.lower()
if not RUN_LAYER3_GATE:
    print("\nRUN_LAYER3_GATE is False — not enforcing")
elif not layer3_ok:
    print("\n⚠️ Gate: install `show_gids` / rdma-core or inspect GIDs manually.")
else:
    print("\n✅ Layer 3 informational check complete — pick NCCL_IB_GID_INDEX to match RoCE v2 row.")

---
## Troubleshooting — zombie VRAM & stray GPU processes

If one node shows **less free memory** than the other for **no obvious reason**, you often have a **zombie** user process, forgotten **Docker** container, or stale **compute** client. **FlashAttention is the victim, not the suspect** — fix memory *before* chasing kernels.

This cell is **read-only** by default: it prints what is holding the GPU. You decide whether to `kill` / `docker stop`.

In [ ]:
def gpu_process_report(label: str, text: str) -> None:
    print(f"\n=== {label} ===")
    print(text.strip() or "(no output)")


if not RUN_ZOMBIE_CHECK:
    print("RUN_ZOMBIE_CHECK is False — skipping")
else:
    r = sh(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"])
    gpu_process_report("Local compute apps", r.stdout + r.stderr)

    r2 = sh(["ssh", WORKER_SSH, "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv"])
    gpu_process_report("Worker compute apps", r2.stdout + r2.stderr)

    print("\n--- Optional: who has /dev/nvidia* open (run with sudo if empty) ---")
    r3 = sh(["fuser", "-v", "/dev/nvidia0"])
    print(r3.stdout + r3.stderr or "(no fuser output; try: sudo fuser -v /dev/nvidia*)")

    print("\n✅ If PIDs are listed, stop those jobs/containers, then re-run Layer 2.")

---
## Layer 4 — eugr `launch-cluster.sh` command builder

Generates the **exact** style of command used in [`docs/playbook-commands.md`](../docs/playbook-commands.md). **Review** before execution.

- Set `RUN_LAUNCH_CLUSTER = True` in the config cell only when you are ready.
- Prefer `cd` into an **expanded** path (shell `eval` or type full path).

In [ ]:
import os

NODES = f"{HEAD_IP},{WORKER_IP}"
spark_repo = os.path.expanduser(SPARK_VLLM_DOCKER)

launch_cmd = (
    f"cd {spark_repo} && "
    f"./launch-cluster.sh -n {NODES} "
    f"--eth-if {ETHERNET_IF} --ib-if {IB_IF} "
    f'-e MODEL_ID={MODEL_ID} '
    f"start"
)

print("Proposed command:\n")
print(launch_cmd)

if RUN_LAUNCH_CLUSTER:
    print("\n--- executing (long-running) ---")
    os.system(launch_cmd)
else:
    print("\n(Set RUN_LAUNCH_CLUSTER = True in the config cell to run.)")

---
## Layer 5 — Live health: logs, VRAM strip chart, smoke `curl`

After vLLM is listening (often `8000` on the head), enable **`RUN_HEALTH_MONITOR`** in the config cell.

- **Logs:** last lines of the container (name may differ).
- **VRAM:** quick matplotlib sample over a few seconds (local + optional SSH worker).
- **Smoke:** `GET /v1/models` against the OpenAI-compatible server.

In [ ]:
import time

import matplotlib.pyplot as plt
import requests


def smi_used_mib() -> float | None:
    r = sh(
        [
            "nvidia-smi",
            "--query-gpu=memory.used",
            "--format=csv,noheader,nounits",
        ]
    )
    if r.returncode != 0:
        return None
    try:
        return float(r.stdout.strip().splitlines()[0].strip())
    except (ValueError, IndexError):
        return None


def remote_smi_used_mib() -> float | None:
    r = sh(
        [
            "ssh",
            WORKER_SSH,
            "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
        ]
    )
    if r.returncode != 0:
        return None
    try:
        return float(r.stdout.strip().splitlines()[0].strip())
    except (ValueError, IndexError):
        return None


if not RUN_HEALTH_MONITOR:
    print("RUN_HEALTH_MONITOR is False — skipping dashboard")
else:
    rlog = sh(["docker", "logs", "--tail", "20", VLLM_CONTAINER_NAME])
    print("=== docker logs (tail) ===")
    print(rlog.stdout + rlog.stderr)

    samples_local: list[float] = []
    samples_remote: list[float] = []
    for _ in range(8):
        samples_local.append(smi_used_mib() or 0.0)
        samples_remote.append(remote_smi_used_mib() or 0.0)
        time.sleep(0.75)

    plt.figure(figsize=(8, 3))
    plt.plot(samples_local, label="head mem used (MiB)")
    plt.plot(samples_remote, label="worker mem used (MiB)")
    plt.xlabel("sample")
    plt.ylabel("MiB")
    plt.legend()
    plt.title("VRAM while loading / serving (coarse)")
    plt.tight_layout()
    plt.show()

    try:
        resp = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        print("\n=== /v1/models ===")
        print(resp.text[:2000])
    except requests.RequestException as e:
        print("\nSmoke request failed:", e)

---
## Next steps

- Materialize weights and `rsync` to the worker — see [playbook-commands.md](../docs/playbook-commands.md).
- If hangs look like **FlashAttention**, re-read [troubleshooting — NCCL first](../docs/troubleshooting-and-pitfalls.md).
- Point agents at the API: [`examples/langgraph-connection.py`](../examples/langgraph-connection.py).